# MLP代码案例

In [ ]:
#定义库
import torch
from torch.utils.data import DataLoader #加载数据集
from torchvision import transforms  #图像转换
from torchvision.datasets import MNIST  #下载数据集
import matplotlib.pyplot as plt
import os

# 定义数据集路径
print("当前终端运行位置:", os.getcwd())
mnist_path = './MNIST/raw'

#神经网络的主体，包括四个全链接层
class Net(torch.nn.Module):

    def __init__(self):
        super().__init__()
        #输入28*28像素的图像
        self.fc1 = torch.nn.Linear(28*28, 64)
        #中间三层，放64个节点，输出为10个数字类别
        self.fc2 = torch.nn.Linear(64, 64)
        self.fc3 = torch.nn.Linear(64, 64)
        self.fc4 = torch.nn.Linear(64, 10)
    
    #forward定义了前向传播过程，参数x是图像输入
    def forward(self, x):
        #self.fc(x)是全连接线性计算
        #torch.nn.functional.relu是激活函数
        x = torch.nn.functional.relu(self.fc1(x))
        x = torch.nn.functional.relu(self.fc2(x))
        x = torch.nn.functional.relu(self.fc3(x))
        #输出层通过softmax归一化，log_softmax是为了提高计算的稳定性
        x = torch.nn.functional.log_softmax(self.fc4(x), dim=1)
        return x





#下载，加载数据集
def get_data_loader(is_train):
    
    #数据转换类型transforms.ToTensor()，tensor是张量
    to_tensor = transforms.Compose([transforms.ToTensor()])
    
    # 3 构建pipeline，对图形做处理
    pipeline = transforms.Compose([
        transforms.ToTensor(), # 将图片转换成tensor
        transforms.Normalize((0.1307,), (0.3081,)) # 正则化：降低模型复杂度
    ])

    #下载数据集，第一个参数为空代表下载到当前目录，is_train用于导入训练集
    #data_set = MNIST("", is_train, transform=to_tensor, download=True)
    # data_set = MNIST("", is_train, transform=pipeline, download=True)


    # 检查文件是否存在
    if not os.path.exists(mnist_path):
        # 如果文件不存在，则下载数据
        # transform = Compose([ToTensor()])
        print("download")
        # train_set = MNIST(root='./', train=True, transform=pipeline, download=True)
        # test_set = MNIST(root='./', train=False, transform=pipeline, download=True)
        
        data_set = MNIST("", is_train, transform=pipeline, download=True)
    else:
    # 如果文件存在，则加载数据
        # transform = Compose([ToTensor()])
        # train_set = MNIST(root='./', train=True, transform=pipeline, download=False)
        # test_set = MNIST(root='./', train=False, transform=pipeline, download=False)
        
        data_set = MNIST("", is_train, transform=pipeline, download=False)


    # 增加判断逻辑：是否已经下载过数据集
    #data_set = MNIST("", is_train, transform=pipeline, download=False)

    #batch_size=15，表示包含15张图片
    #shuffle=True，表示数据是随机打乱的
    #return DataLoader返回数据加载器
    return DataLoader(data_set, batch_size=15, shuffle=True)

#判别神经网络的识别正确率
def evaluate(test_data, net):
    n_correct = 0
    n_total = 0
    with torch.no_grad():
        #从测试集中批次测试数据
        for (x, y) in test_data:
            #计算神经网络的预测值
            outputs = net.forward(x.view(-1, 28*28))
            #对批次中的每个结果进行比较
            for i, output in enumerate(outputs):
                #argmax函数计算一个数列中最大值的序号，就是预测的手写数字预测结果
                if torch.argmax(output) == y[i]:
                    n_correct += 1
                n_total += 1
    #最后函数返回正确值
    return n_correct / n_total


def main():
    #主函数中，先导入训练集和测试集，初始化神经网络
    train_data = get_data_loader(is_train=True)
    test_data = get_data_loader(is_train=False)
    net = Net()
    
    #在训练开始之前，先打印初始网络的正确率，接近1/10
    print("initial accuracy:", evaluate(test_data, net))
    
    #在训练神经网络，都是pytorch的固定写法
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    for epoch in range(2):  #反复训练神经网络几个轮次
        for (x, y) in train_data:
            net.zero_grad() #初始化
            output = net.forward(x.view(-1, 28*28)) #正向传播
            #nll_loss是对数损失函数，是为了匹配前面log_softmax中的对数运算
            loss = torch.nn.functional.nll_loss(output, y)  #计算差值
            loss.backward()    #反向误差传播
            optimizer.step()    #优化网络参数
        
        #打印当前网络的正确率
        print("epoch", epoch, "accuracy:", evaluate(test_data, net))

    #显示损失和准确度
    for (n, (x, _)) in enumerate(test_data):
        if n > 3:
            break
        predict = torch.argmax(net.forward(x[0].view(-1, 28*28)))
        plt.figure(n)
        plt.imshow(x[0].view(28, 28))
        plt.title("prediction: " + str(int(predict)))
    plt.show()


if __name__ == "__main__":
    main()
